# 27.4 Embodied AI & robotics learning

Embodied AI closes the loop: perception, action, reward, and physics all answer back. This notebook simulates TD learning, softmax exploration, imitation, feedback control, and sim-to-real shift on CPU-only toy dynamics. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(274)


def softmax(x, beta=1.0):
    z = beta * np.asarray(x, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def success_rate(successes):
    return float(np.mean(np.asarray(successes) > 0))

## The concept, built once (D1)

A one-step TD update uses $\delta_t=r_t+\gamma V(s_{t+1})-V(s_t)$ and $V(s_t)\leftarrow V(s_t)+\eta\delta_t$. The lesson values give target $1.350$, error $0.350$, and updated value $1.070$.

In [ ]:
def closed_loop_agent_step(state, action, reward, value, next_value, gamma=0.9, eta=0.2):
    target = reward + gamma * next_value
    error = target - value
    updated_value = value + eta * error
    return {"state": state, "action": action, "target": target, "error": error, "updated_value": updated_value}

lesson_step = closed_loop_agent_step("s", "move", 0.0, 1.0, 1.5)

assert round(lesson_step["target"], 3) == 1.350
assert round(lesson_step["error"], 3) == 0.350
assert round(lesson_step["updated_value"], 3) == 1.070

print(lesson_step)

Exploration can be soft rather than greedy. With $Q=[1.2,0.7,0.4]$ and $\beta=2$, the lesson policy is $[0.637,0.234,0.129]$. Imitation and feedback checks give loss $0.0324$, sim-real gap $0.190$, and three-step error $2.160$ cm.

In [ ]:
q_values = np.array([1.2, 0.7, 0.4])
policy = softmax(q_values, beta=2.0)
bc_loss = (0.62 - 0.80) ** 2
sim_success = 92 / 100
real_success = 73 / 100
gap = sim_success - real_success
error_after_three = 10 * 0.6 ** 3

assert np.round(policy, 3).tolist() == [0.637, 0.234, 0.129]
assert round(bc_loss, 4) == 0.0324
assert round(gap, 3) == 0.190
assert round(error_after_three, 3) == 2.160

print("softmax policy", np.round(policy, 3))
print("behavior cloning loss", round(bc_loss, 4))
print("sim-real gap", round(gap, 3))
print("feedback error", round(error_after_three, 3))

## The dataset ladder

The ladder rises from a 2-state MDP to 1D reaching, noisy demonstrations with recovery states, a gridworld with visual-like state features, and a sim-to-real shifted friction instance.

In [ ]:
def make_embodied_ladder(seed=274):
    local_rng = np.random.default_rng(seed)
    ladder = []

    ladder.append({"name": "D1 TD toy", "positions": np.array([0.0, 1.0]), "goals": np.array([1.0, 1.0]), "friction": 1.0, "noise": 0.0, "complexity": 1})

    positions = np.linspace(-1.0, 1.0, 12)
    goals = np.zeros_like(positions)
    ladder.append({"name": "D2 1D reaching", "positions": positions, "goals": goals, "friction": 1.0, "noise": 0.0, "complexity": 2})

    positions = local_rng.uniform(-1.5, 1.5, size=24)
    goals = np.zeros_like(positions)
    ladder.append({"name": "D3 noisy demos", "positions": positions, "goals": goals, "friction": 0.9, "noise": 0.08, "complexity": 4})

    xs = local_rng.integers(0, 5, size=30)
    ys = local_rng.integers(0, 5, size=30)
    positions = np.column_stack([xs, ys]).astype(float)
    goals = np.tile(np.array([4.0, 4.0]), (30, 1))
    ladder.append({"name": "D4 visual gridworld", "positions2": positions, "goals2": goals, "friction": 1.0, "noise": 0.05, "complexity": 6})

    positions = local_rng.uniform(-1.2, 1.2, size=100)
    goals = np.zeros_like(positions)
    ladder.append({"name": "D5 shifted friction", "positions": positions, "goals": goals, "friction": 0.55, "noise": 0.10, "complexity": 10})

    return ladder

embodied_ladder = make_embodied_ladder()

for rung in embodied_ladder:
    if "positions2" in rung:
        shape = rung["positions2"].shape
        sample = rung["positions2"][:3]
    else:
        shape = rung["positions"].shape
        sample = np.round(rung["positions"][:3], 3)
    print(rung["name"], "shape", shape, "friction", rung["friction"], "sample", sample)

In [ ]:
def run_embodied_rung(rung, correction=0.45, randomize=False):
    local_rng = np.random.default_rng(100 + int(rung["complexity"]))
    trajectories = []
    successes = []
    returns = []
    if "positions2" in rung:
        starts = rung["positions2"].copy()
        goals = rung["goals2"]
    else:
        starts = rung["positions"].copy()
        goals = rung["goals"]
    for i, start in enumerate(starts):
        state = np.asarray(start, dtype=float)
        goal = np.asarray(goals[i], dtype=float)
        path = [state.copy()]
        total_return = 0.0
        for step in range(8):
            friction = rung["friction"]
            if randomize:
                friction = local_rng.uniform(0.5, 1.05)
            action = correction * (goal - state)
            noise = local_rng.normal(0.0, rung["noise"], size=state.shape)
            state = state + friction * action + noise
            distance = float(np.linalg.norm(goal - state))
            total_return += -distance
            path.append(state.copy())
        success = int(float(np.linalg.norm(goal - state)) < 0.25)
        trajectories.append(np.asarray(path))
        successes.append(success)
        returns.append(total_return)
    return {"success_rate": success_rate(successes), "return": float(np.mean(returns)), "trajectories": trajectories, "successes": np.asarray(successes)}

embodied_results = []

for rung in embodied_ladder:
    result = run_embodied_rung(rung)
    embodied_results.append(result)
    print(f"{rung['name']}: success_rate={result['success_rate']:.3f} return={result['return']:.3f}")

print("no-feedback baseline success", 0.0)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))

for i, rung in enumerate(embodied_ladder):
    ax = axes[0, i]
    for traj in embodied_results[i]["trajectories"][:6]:
        if traj.ndim == 2 and traj.shape[1] == 2:
            ax.plot(traj[:, 0], traj[:, 1], alpha=0.7)
        else:
            ax.plot(traj.reshape(-1), alpha=0.7)
    ax.set_title(rung["name"])

complexity = [rung["complexity"] for rung in embodied_ladder]
successes = [result["success_rate"] for result in embodied_results]
axes[1, 0].plot(complexity, successes, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("Success vs dynamics complexity")
axes[1, 0].set_xlabel("complexity")
axes[1, 0].set_ylabel("success rate")

for j in range(1, 5):
    axes[1, j].bar(["success", "failure"], [successes[j], 1 - successes[j]])
    axes[1, j].set_ylim(0, 1)
    axes[1, j].set_title(f"D{j + 1} outcomes")

plt.tight_layout()

## Pitfall on the hardest rung: ignoring sim-to-real gaps

The lesson-style gap is 92/100 simulation successes versus 73/100 hardware successes, a 0.190 drop. Here the shifted friction version reproduces the same idea, then domain randomization plus feedback correction improves transfer.

In [ ]:
d5 = embodied_ladder[-1]
sim_like = dict(d5)
sim_like["friction"] = 1.0
sim_result = run_embodied_rung(sim_like, correction=0.45)
real_result = run_embodied_rung(d5, correction=0.45)
fixed_result = run_embodied_rung(d5, correction=0.65, randomize=True)
lesson_gap = 0.920 - 0.730

assert round(lesson_gap, 3) == 0.190

print("sim success", round(sim_result["success_rate"], 3))
print("shifted real success", round(real_result["success_rate"], 3))
print("domain-randomized feedback success", round(fixed_result["success_rate"], 3))
print("lesson gap", round(lesson_gap, 3))

## Evaluate it + Practice
- Compare the rung metric against the no-skill baseline printed above.
- Run one cheap sanity check: change one label, threshold, route, or floor and confirm the metric moves in the expected direction.
- Ablate the key idea by turning off the kernel, leak, tool verification, feedback, or safety gate.
- Watch failure signals: flat metric curves, hidden weak coordinates, high cost, or a D5 pitfall that does not improve after the fix.

Practice prompts:
1. Change the D3 noise or shift level and predict which rung changes most.

2. Replace the baseline with a stronger classical or rule-based check and compare the metric.

3. Add one new D5 stress case and explain whether the pitfall fix still works.